<a href="https://colab.research.google.com/github/sumit-teotia/ppe-detection-yolo26/blob/main/pp_detection_yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U ultralytics kaggle

from ultralytics import YOLO

model = YOLO("yolo26n.pt")

print("✅ Ultralytics ready")
print("✅ YOLO26n loaded")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 8.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
✅ Ultralytics ready
✅ YOLO26n loaded


In [ ]:
import os

KAGGLE_TOKEN = "KGAT_8c20188a9f43ad06f4f61f8efdd35fc8"

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN

print("✅ Kaggle token set")

✅ Kaggle token set


In [ ]:
!kaggle datasets download \
    -d ndomalau/personal-protective-equipment-ppe-dataset \
    -p /content/ppe_dataset \
    --unzip

print("✅ PPE dataset downloaded and extracted")

Dataset URL: https://www.kaggle.com/datasets/ndomalau/personal-protective-equipment-ppe-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 238M/238M [00:03<00:00, 79.7MB/s]

✅ PPE dataset downloaded and extracted


In [ ]:
from pathlib import Path
import yaml

DATASET_DIR = Path("/content/ppe_dataset")

# Find data.yaml
yaml_path = next(DATASET_DIR.rglob("data.yaml"))

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

DATASET_ROOT = yaml_path.parent

print("✅ Dataset found")
print("📂 Location:", DATASET_ROOT)

print("\n📦 Classes:")
for class_id, class_name in enumerate(data["names"]):
    print(f"{class_id}: {class_name}")

print("\n📊 Image counts:")

for split in ["train", "val", "test"]:
    image_dir = DATASET_ROOT / split / "images"

    if image_dir.exists():
        count = sum(
            1 for f in image_dir.iterdir()
            if f.suffix.lower() in [
                ".jpg", ".jpeg", ".png", ".bmp", ".webp"
            ]
        )
        print(f"{split}: {count} images")
    else:
        print(f"{split}: folder not found")

✅ Dataset found
📂 Location: /content/ppe_dataset/Personal-Protective-Equipment (PPE) Dataset

📦 Classes:
0: helmet
1: no_helmet
2: no_vest
3: person
4: vest

📊 Image counts:
train: 3248 images
val: folder not found
test: 406 images


In [ ]:
from pathlib import Path
import shutil
import random

TRAIN_IMAGES = DATASET_ROOT / "train" / "images"
TRAIN_LABELS = DATASET_ROOT / "train" / "labels"

VAL_IMAGES = DATASET_ROOT / "val" / "images"
VAL_LABELS = DATASET_ROOT / "val" / "labels"

VAL_IMAGES.mkdir(parents=True, exist_ok=True)
VAL_LABELS.mkdir(parents=True, exist_ok=True)

images = [
    f for f in TRAIN_IMAGES.iterdir()
    if f.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
]

print("Original training images:", len(images))

random.seed(42)
random.shuffle(images)

val_count = int(len(images) * 0.20)
val_images = images[:val_count]

print("Moving to validation:", len(val_images))

moved = 0

for image in val_images:

    label = TRAIN_LABELS / f"{image.stem}.txt"

    if not label.exists():
        print("⚠️ Label missing:", image.name)
        continue

    shutil.move(str(image), str(VAL_IMAGES / image.name))
    shutil.move(str(label), str(VAL_LABELS / label.name))

    moved += 1

print("\n✅ Validation split complete")
print("Train images remaining:", len(images) - moved)
print("Validation images:", moved)

Original training images: 3248
Moving to validation: 649

✅ Validation split complete
Train images remaining: 2599
Validation images: 649


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/PPE_YOLO_Project")
DRIVE_DATASET = DRIVE_PROJECT / "dataset"
DRIVE_MODEL = DRIVE_PROJECT / "model"

DRIVE_DATASET.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL.mkdir(parents=True, exist_ok=True)

print("✅ Google Drive connected")
print("📂 Project:", DRIVE_PROJECT)
print("📦 Dataset backup:", DRIVE_DATASET)
print("🤖 Model backup:", DRIVE_MODEL)

Mounted at /content/drive
✅ Google Drive connected
📂 Project: /content/drive/MyDrive/PPE_YOLO_Project
📦 Dataset backup: /content/drive/MyDrive/PPE_YOLO_Project/dataset
🤖 Model backup: /content/drive/MyDrive/PPE_YOLO_Project/model


In [ ]:
import shutil
from pathlib import Path

SOURCE_DATASET = DATASET_ROOT
BACKUP_DATASET = DRIVE_DATASET / SOURCE_DATASET.name

if BACKUP_DATASET.exists():
    shutil.rmtree(BACKUP_DATASET)

print("📦 Copying dataset to Google Drive...")

shutil.copytree(
    SOURCE_DATASET,
    BACKUP_DATASET
)

print("✅ Dataset backup complete!")
print("📂 Saved at:", BACKUP_DATASET)

# Verify
print("\nTrain:", (BACKUP_DATASET / "train" / "images").exists())
print("Val:", (BACKUP_DATASET / "val" / "images").exists())
print("Test:", (BACKUP_DATASET / "test" / "images").exists())
print("YAML:", (BACKUP_DATASET / "data.yaml").exists())

📦 Copying dataset to Google Drive...
✅ Dataset backup complete!
📂 Saved at: /content/drive/MyDrive/PPE_YOLO_Project/dataset/Personal-Protective-Equipment (PPE) Dataset

Train: True
Val: True
Test: True
YAML: True


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data=str(DATASET_ROOT / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    project="/content/ppe_project",
    name="ppe_detection",
    plots=True
)

print("✅ Training complete!")

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ppe_dataset/Personal-Protective-Equipment (PPE) Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe_detection,

In [ ]:
from pathlib import Path
import shutil

# Training output
RUN_DIR = Path("/content/ppe_project/ppe_detection")
BEST_MODEL = RUN_DIR / "weights" / "best.pt"
LAST_MODEL = RUN_DIR / "weights" / "last.pt"

# Google Drive model folder
DRIVE_MODEL = Path("/content/drive/MyDrive/PPE_YOLO_Project/model")
DRIVE_MODEL.mkdir(parents=True, exist_ok=True)

# Copy trained models
shutil.copy2(BEST_MODEL, DRIVE_MODEL / "best.pt")
shutil.copy2(LAST_MODEL, DRIVE_MODEL / "last.pt")

print("✅ Training completed")
print("✅ best.pt saved to:", DRIVE_MODEL / "best.pt")
print("✅ last.pt saved to:", DRIVE_MODEL / "last.pt")

✅ Training completed
✅ best.pt saved to: /content/drive/MyDrive/PPE_YOLO_Project/model/best.pt
✅ last.pt saved to: /content/drive/MyDrive/PPE_YOLO_Project/model/last.pt


In [ ]:
model = YOLO("/content/drive/MyDrive/PPE_YOLO_Project/model/best.pt")

print("✅ Trained PPE model loaded successfully")
print(model.names)

✅ Trained PPE model loaded successfully
{0: 'helmet', 1: 'no_helmet', 2: 'no_vest', 3: 'person', 4: 'vest'}


In [ ]:
from ultralytics import YOLO

MODEL_PATH = "/content/drive/MyDrive/PPE_YOLO_Project/model/best.pt"

model = YOLO(MODEL_PATH)

metrics = model.val(
    data=str(DATASET_ROOT / "data.yaml"),
    imgsz=640,
    batch=16,
    device=0,
    plots=True
)

print("\n✅ Evaluation completed!")
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,375,811 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1893.7±502.5 MB/s, size: 63.7 KB)
val: Scanning /content/ppe_dataset/Personal-Protective-Equipment (PPE) Dataset/valid/labels.cache... 406 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 406/406 131.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 3.5it/s 7.5s
                   all        406       3095      0.912      0.853      0.916      0.597
                helmet        246        531      0.912      0.842      0.911      0.578
             no_helmet        241        523      0.901      0.788      0.876      0.508
               no_vest        269        649      0.879      0.838      0.885      0.513
                person        406       1042      0.921      0.902      0.947      0.

In [1]:
# Approximate detection accuracy
# Based on Precision and Recall

precision = 0.9124859739121252
recall = 0.8533887038262078

accuracy = ((precision + recall) / 2) * 100

print(f"Approximate Accuracy: {accuracy:.2f}%")

Approximate Accuracy: 88.29%
